In [19]:
import pandas as pd
import networkx as nx

# Simple transaction network: who sent money to whom
transactions = pd.DataFrame({
    'sender': ['Alice', 'Bob', 'Charlie', 'Alice', 'David', 'Eve', 'Frank'],
    'receiver': ['Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Frank', 'George'],
    'amount': [100, 500, 200, 150, 300, 250, 1000]
})

print(transactions)

# RED FLAGS
# Long chain of transfers (money launderng)
# Circular transfers (suspicious loops)
# Hub Nodes (Someone receiving form many people)

G = nx.from_pandas_edgelist(
    transactions,
    source='sender',
    target='receiver',
    edge_attr='amount',
    create_using=nx.DiGraph()
)

# Basic graph info
print(f"Nodes (people): {G.nodes()}")
print(f"Edges (transactions): {G.edges()}")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print()

    sender receiver  amount
0    Alice      Bob     100
1      Bob  Charlie     500
2  Charlie    David     200
3    Alice      Eve     150
4    David    Frank     300
5      Eve    Frank     250
6    Frank   George    1000
Nodes (people): ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'George']
Edges (transactions): [('Alice', 'Bob'), ('Alice', 'Eve'), ('Bob', 'Charlie'), ('Charlie', 'David'), ('David', 'Frank'), ('Eve', 'Frank'), ('Frank', 'George')]
Number of nodes: 7
Number of edges: 7



In [18]:
# Who has the most outgoing transactions (senders)?
# Out-degree: how many people someone sent money to (activity level)

out_degree = dict(G.out_degree())
print("Outgoing transactions (sender activity):")
print(sorted(out_degree.items(), key=lambda x: x[1], reverse=True))
print()

# 1. out_degree.items() — convert dict to list of [(key, value)] tuples
# 2. key=lambda x: x[1] — sort by the value (second element, index 1)

# --------- #

# Who has the most incoming transactions (receivers)?
# In-degree: how many people sent money to someone (popularity/sink)

in_degree = dict(G.in_degree())
print("Incoming transactions (receiver activity):")
print(sorted(in_degree.items(), key=lambda x: x[1], reverse=True))

Outgoing transactions (sender activity):
[('Alice', 2), ('Bob', 1), ('Charlie', 1), ('David', 1), ('Eve', 1), ('Frank', 1), ('George', 0)]

Incoming transactions (receiver activity):
[('Frank', 2), ('Bob', 1), ('Charlie', 1), ('David', 1), ('Eve', 1), ('George', 1), ('Alice', 0)]


> In fraud: high in-degree + high out-degree together = suspicious hub.

In [ ]:
# Detect cycles - suspicious pattern in fraud
cycles = list(nx.simple_cycles(G))
print(f"Circular transfers detected: {len(cycles)}")
for cycle in cycles:
    print(f"  Cycle: {' → '.join(cycle)} → {cycle[0]}")
print()

# Example: if Alice → Bob → Charlie → Alice, 
# money goes in a circle (money laundering red flag)
# Cycles: Circular money flows (fraud indicator)

#-------------#

# Check if graph is acyclic (no cycles at all)
is_acyclic = nx.is_directed_acyclic_graph(G)
print(f"Is graph acyclic (no cycles)? {is_acyclic}")
print()

# Find all paths from one person to another
# Example: all ways money can flow from Alice to George
if 'Alice' in G and 'George' in G:
    paths = list(nx.all_simple_paths(G, 'Alice', 'George'))
    print(f"Paths from Alice to George:")
    for path in paths:
        print(f"  {' → '.join(path)}")

# All paths: How money can flow through the network (tracing suspicious chains)

Circular transfers detected: 0

Is graph acyclic (no cycles)? True

Paths from Alice to George:
  Alice → Bob → Charlie → David → Frank → George
  Alice → Eve → Frank → George


In [16]:
# Betweenness centrality: who sits in the middle of many paths?
# High value = person is a "middleman" (suspicious in fraud)
betweenness = nx.betweenness_centrality(G)
print("Betweenness Centrality (middleman score):")
for person, score in sorted(betweenness.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.3f}")
print()

# Closeness centrality: who is closest to everyone?
# High value = person is well-connected (hub)
closeness = nx.closeness_centrality(G)
print("Closeness Centrality (connectivity hub):")
for person, score in sorted(closeness.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.3f}")

Betweenness Centrality (middleman score):
  Frank: 0.167
  Charlie: 0.133
  David: 0.133
  Bob: 0.067
  Eve: 0.067
  Alice: 0.000
  George: 0.000

Closeness Centrality (connectivity hub):
  Frank: 0.463
  George: 0.400
  David: 0.250
  Charlie: 0.222
  Bob: 0.167
  Eve: 0.167
  Alice: 0.000


In [13]:
# PageRank: influence based on incoming connections
pagerank = nx.pagerank(G, weight='amount')
print("PageRank (influence/authority):")
for person, rank in sorted(pagerank.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {rank:.4f}")
print()

# Who receives the most total money (weighted in-degree)?
weighted_in_degree = dict(G.in_degree(weight='amount'))
print("Weighted In-Degree (total money received):")
for person, amount in sorted(weighted_in_degree.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: ${amount}")


PageRank (influence/authority):
  George: 0.2696
  Frank: 0.2534
  David: 0.1526
  Charlie: 0.1159
  Eve: 0.0818
  Bob: 0.0726
  Alice: 0.0542

Weighted In-Degree (total money received):
  George: $1000
  Frank: $550
  Charlie: $500
  David: $200
  Eve: $150
  Bob: $100
  Alice: $0


In [14]:
# Ego graph: all people directly connected to Frank
ego_frank = nx.ego_graph(G, 'Frank')
print(f"People directly connected to Frank:")
print(f"  Nodes: {list(ego_frank.nodes())}")
print(f"  Edges: {list(ego_frank.edges())}")
print()

# Predecessors (who sent money to George)?
george_senders = list(G.predecessors('George'))
print(f"Who sent money to George? {george_senders}")
print()

# Successors (who did Frank send money to)?
frank_receivers = list(G.successors('Frank'))
print(f"Who did Frank send money to? {frank_receivers}")
print()

# Summary: create a suspicious subgraph
# (everyone who connects to George within 1 hop)
suspicious_subgraph = ego_frank.copy()
print(f"Suspicious network around Frank:")
print(f"  Involved: {suspicious_subgraph.nodes()}")
print(f"  Transactions: {suspicious_subgraph.edges()}")


People directly connected to Frank:
  Nodes: ['George', 'Frank']
  Edges: [('Frank', 'George')]

Who sent money to George? ['Frank']

Who did Frank send money to? ['George']

Suspicious network around Frank:
  Involved: ['George', 'Frank']
  Transactions: [('Frank', 'George')]


In [15]:
# Composite fraud score: combine multiple signals
fraud_score = {}

for node in G.nodes():
    score = 0
    
    # Signal 1: Betweenness (middleman activity)
    score += betweenness[node] * 100
    
    # Signal 2: High in-degree + out-degree (hub)
    in_deg = G.in_degree(node)
    out_deg = G.out_degree(node)
    if in_deg > 1 and out_deg > 1:
        score += 50
    
    # Signal 3: PageRank (influence)
    score += pagerank[node] * 50
    
    # Signal 4: Money concentration (lots in, lots out)
    in_amount = weighted_in_degree.get(node, 0)
    out_amount = dict(G.out_degree(weight='amount')).get(node, 0)
    if in_amount > 300 and out_amount > 0:
        score += 30
    
    fraud_score[node] = score

print("Fraud Suspicion Score (higher = more suspicious):")
for person, score in sorted(fraud_score.items(), key=lambda x: x[1], reverse=True):
    print(f"  {person}: {score:.2f}")


Fraud Suspicion Score (higher = more suspicious):
  Frank: 59.34
  Charlie: 49.13
  David: 20.97
  George: 13.48
  Eve: 10.76
  Bob: 10.30
  Alice: 2.71
